### Rademacher complexity

Let $\mathcal{F}$ be a family of functions $f : \mathcal{X} \to \mathbb{R}$ and let $S = (x_1, \ldots, x_n)$ be a sample. The empirical Rademacher complexity of $\mathcal{F}$ on $S$ is

$$
\widehat{\mathfrak{R}}_S(\mathcal{F}) = \mathbb{E}_{\sigma}\left[\sup_{f \in \mathcal{F}} \frac{1}{n} \sum_{i=1}^n \sigma_i f(x_i)\right],
$$

where $\sigma_1, \ldots, \sigma_n$ are independent Rademacher random variables, that is $\mathbb{P}(\sigma_i = 1) = \mathbb{P}(\sigma_i = -1) = 1/2$.

Intuitively, the empirical Rademacher complexity measures how well the class $\mathcal{F}$ can fit random signs on the observed sample. A richer function class should typically have larger complexity.

In this part, we will study this numerically for simple binary-valued function classes on $[0,1]$.

#### Threshold and interval classes

We consider the following function classes:

- **Thresholds**

$$
\mathcal{F}_{\mathrm{thr}} = \{f_t(x) = \mathbf{1}\{x \le t\} : t \in [0,1]\}
$$

- **Intervals**

$$
\mathcal{F}_{\mathrm{int}} = \{f_{a,b}(x) = \mathbf{1}\{a \le x \le b\} : 0 \le a \le b \le 1\}
$$

On a finite sample $S = (x_1, \ldots, x_n)$, both classes induce only finitely many effective hypotheses. In practice, it is enough to evaluate the functions on thresholds or interval endpoints determined by the sample points.

We will estimate the empirical Rademacher complexity by Monte Carlo:

$$
\widehat{\mathfrak{R}}_S(\mathcal{F}) \approx \frac{1}{M} \sum_{m=1}^M \sup_{f \in \mathcal{F}} \frac{1}{n} \sum_{i=1}^n \sigma_i^{(m)} f(x_i).
$$

**Task 1:** Let $x_1, \ldots, x_n \sim U(0,1)$ and consider the threshold class $\mathcal{F}_{\mathrm{thr}}$.

Write code that:

1. generates a sample $S$ of size $n$,
2. approximates $\widehat{\mathfrak{R}}_S(\mathcal{F}_{\mathrm{thr}})$ by Monte Carlo,
3. plots the estimate as a function of $n$ for $n = 10, 20, \ldots, 200$.

**Hint:** After sorting the sample, the threshold functions correspond to taking prefixes of the sorted sample.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

def threshold_supremum(sigmas):
    # Effective hypotheses are prefixes of the sorted sample.
    prefix_sums = np.concatenate([[0], np.cumsum(sigmas)])
    return np.max(prefix_sums) / len(sigmas)

def empirical_rademacher_threshold(n, M=2000, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    x = np.sort(rng.uniform(0, 1, size=n))
    vals = []
    for _ in range(M):
        sigmas = rng.choice([-1, 1], size=n)
        vals.append(threshold_supremum(sigmas))
    return np.mean(vals), x

n_values = np.arange(10, 201, 10)
rads_thr = []

for n in n_values:
    rad_hat, _ = empirical_rademacher_threshold(n, M=2000, rng=rng)
    rads_thr.append(rad_hat)

plt.figure(figsize=(7, 4))
plt.plot(n_values, rads_thr, marker="o", label="Threshold class")
plt.xlabel("n")
plt.ylabel("Estimated empirical Rademacher complexity")
plt.title("Threshold class on U(0,1) samples")
plt.grid(True)
plt.legend()
plt.show()

**Answer:** The estimated empirical Rademacher complexity decreases with $n$. This is expected: as the sample size grows, fitting random signs becomes harder on average, so the normalized supremum becomes smaller.

**Task 2:** Repeat the experiment for the interval class $\mathcal{F}_{\mathrm{int}}$ and compare it with the threshold class on the same figure.

Do you observe that the interval class has larger empirical Rademacher complexity? Does the complexity tend to decrease as $n$ grows?

**Hint:** After sorting the sample, interval functions correspond to selecting contiguous blocks of the sorted points.

In [ ]:
def interval_supremum(sigmas):
    # Effective hypotheses are contiguous blocks of the sorted sample.
    best = 0
    current = 0
    for s in sigmas:
        current = max(0, current + s)
        best = max(best, current)
    return best / len(sigmas)

def empirical_rademacher_interval(n, M=2000, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    x = np.sort(rng.uniform(0, 1, size=n))
    vals = []
    for _ in range(M):
        sigmas = rng.choice([-1, 1], size=n)
        vals.append(interval_supremum(sigmas))
    return np.mean(vals), x

rads_int = []
for n in n_values:
    rad_hat, _ = empirical_rademacher_interval(n, M=2000, rng=rng)
    rads_int.append(rad_hat)

plt.figure(figsize=(7, 4))
plt.plot(n_values, rads_thr, marker="o", label="Threshold class")
plt.plot(n_values, rads_int, marker="s", label="Interval class")
plt.xlabel("n")
plt.ylabel("Estimated empirical Rademacher complexity")
plt.title("Threshold vs interval classes")
plt.grid(True)
plt.legend()
plt.show()

**Answer:** Yes, the interval class has larger empirical Rademacher complexity than the threshold class. This matches intuition, since intervals form a richer class than prefixes. Both curves decrease with $n$, although the interval class stays above the threshold class.

**Task 3:** Fix one sample size, for example $n = 50$, and repeat the Monte Carlo experiment many times with newly generated samples.

Plot a histogram of the empirical Rademacher complexity for:

- the threshold class,
- the interval class.

Comment on the variability across samples.

In [ ]:
n = 50
B = 300

thr_samples = []
int_samples = []

for _ in range(B):
    val_thr, _ = empirical_rademacher_threshold(n, M=1000, rng=rng)
    val_int, _ = empirical_rademacher_interval(n, M=1000, rng=rng)
    thr_samples.append(val_thr)
    int_samples.append(val_int)

plt.figure(figsize=(7, 4))
plt.hist(thr_samples, bins=20, alpha=0.7, label="Threshold class")
plt.hist(int_samples, bins=20, alpha=0.7, label="Interval class")
plt.xlabel("Estimated empirical Rademacher complexity")
plt.ylabel("Frequency")
plt.title("Variability across samples (n=50)")
plt.legend()
plt.grid(True)
plt.show()

**Answer:** The interval class has a histogram shifted to the right, again indicating larger complexity. The variability across samples is present but moderate. In this experiment, the class effect is stronger than the sample-to-sample fluctuation.

### Time series decomposition

Let $(X_t)_{t=1}^T$ be a time series. A classical additive decomposition writes

$$
X_t = T_t + S_t + R_t,
$$

where:

- $T_t$ is the **trend** component,
- $S_t$ is the **seasonal** component,
- $R_t$ is the **remainder** (or residual) component.

In this lab you will implement a simple decomposition from scratch.

#### Moving averages

A moving average is often used to estimate the trend. For an odd window size $m = 2h+1$, the centered moving average is

$$
\hat{T}_t = \frac{1}{m} \sum_{j=-h}^{h} X_{t+j},
$$

for all indices $t$ such that the full window is available.

Near the boundaries, if $X_{t+j}$ is not defined, you may replace it by $0$.

**Task 4:** Write a function that computes the centered moving average for a given odd window size $q$.

Test your function on a simple synthetic example and plot the original series together with the moving average.

In [ ]:
import pandas as pd

def moving_average_zero_padding(x, q):
    if q % 2 == 0:
        raise ValueError("Window size q must be odd.")
    x = np.asarray(x, dtype=float)
    h = q // 2
    T = len(x)
    ma = np.empty(T)
    for t in range(T):
        total = 0.0
        for j in range(-h, h + 1):
            idx = t + j
            if 0 <= idx < T:
                total += x[idx]
        ma[t] = total / q
    return ma

T = 120
t = np.arange(T)
x = 0.05 * t + 2 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 0.5, size=T)

ma = moving_average_zero_padding(x, q=13)

plt.figure(figsize=(9, 4))
plt.plot(t, x, label="Original series")
plt.plot(t, ma, linewidth=2, label="Centered moving average")
plt.xlabel("t")
plt.ylabel("Value")
plt.title("Synthetic series and moving average")
plt.grid(True)
plt.legend()
plt.show()

**Answer:** The moving average smooths out the short-term oscillations and gives a reasonable estimate of the trend. Because of the zero-padding convention, the estimate is slightly biased near the boundaries.

#### Seasonal component

Assume that the period $p$ is known. Once an estimate $T_t$ of the trend has been computed, define the detrended series

$$
D_t = X_t - \hat{T}_t.
$$

A simple estimate of the seasonal pattern is obtained by averaging the detrended observations with the same position inside the period. For $k \in \{0, 1, \ldots, p-1\}$,

$$
\tilde{S}_k = \frac{1}{N_k} \sum_{t : (t-1) \bmod p = k} D_t.
$$

To make the additive decomposition identifiable, it is common to center the seasonal pattern so that

$$
\sum_{k=0}^{p-1} \hat{S}_k = 0,
$$

which leads to

$$
\hat{S}_k = \tilde{S}_k - \frac{1}{p} \sum_{j=0}^{p-1} \tilde{S}_j.
$$

Then we repeat this pattern over time to obtain $\hat{S}_t$.

**Task 5:** Write a function that, given a time series, a trend estimate, and a period $p$, computes:

1. the seasonal pattern of length $p$,
2. the seasonal component $(\hat{S}_t)_{t=1}^T$ repeated over time,
3. the remainder $(R_t)_{t=1}^T = X_t - \hat{T}_t - \hat{S}_t$.

Test your code on a synthetic time series with known trend and seasonality.

In [ ]:
def seasonal_decomposition_additive(x, trend, p):
    x = np.asarray(x, dtype=float)
    trend = np.asarray(trend, dtype=float)
    T = len(x)
    detrended = x - trend

    seasonal_pattern = np.zeros(p)
    counts = np.zeros(p, dtype=int)

    for t in range(T):
        k = t % p
        seasonal_pattern[k] += detrended[t]
        counts[k] += 1

    seasonal_pattern = seasonal_pattern / counts
    seasonal_pattern = seasonal_pattern - seasonal_pattern.mean()

    seasonal = np.array([seasonal_pattern[t % p] for t in range(T)])
    remainder = x - trend - seasonal

    return seasonal_pattern, seasonal, remainder

T = 120
t = np.arange(T)
true_trend = 0.03 * t
true_pattern = np.array([2.0, 1.5, 0.5, -0.5, -1.5, -2.0, -1.2, -0.4, 0.2, 0.8, 1.3, 1.8])
true_pattern = true_pattern - true_pattern.mean()
true_seasonal = np.array([true_pattern[i % 12] for i in range(T)])
x = true_trend + true_seasonal + rng.normal(0, 0.4, size=T)

trend_hat = moving_average_zero_padding(x, q=13)
pattern_hat, seasonal_hat, remainder_hat = seasonal_decomposition_additive(x, trend_hat, p=12)

fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)
axes[0].plot(x)
axes[0].set_title("Original series")
axes[1].plot(trend_hat)
axes[1].set_title("Estimated trend")
axes[2].plot(seasonal_hat)
axes[2].set_title("Estimated seasonal component")
axes[3].plot(remainder_hat)
axes[3].set_title("Estimated remainder")
for ax in axes:
    ax.grid(True)
plt.tight_layout()
plt.show()

print("Estimated seasonal pattern:")
print(np.round(pattern_hat, 3))

**Answer:** The decomposition recovers the general structure well: the estimated trend is smooth, the estimated seasonal component is periodic, and the remainder looks much less structured than the original signal. The reconstruction is not perfect because of noise and the approximate trend estimate.

#### Autocorrelation function

For a centered time series $(X_t)_{t=1}^T$, the sample autocorrelation at lag $h$ is defined by

$$
\rho(h) = \frac{\sum_{t=1}^{T-h} (X_t - \bar{X})(X_{t+h} - \bar{X})}{\sum_{t=1}^{T} (X_t - \bar{X})^2},
\qquad h = 0,1,\ldots
$$

where $\bar{X}$ is the sample mean.

The autocorrelation function can help detect seasonality: peaks at regular lags often indicate a seasonal period.

**Task 6:** Write a function that computes the sample autocorrelation function up to a chosen maximum lag.

Check your implementation on a synthetic seasonal time series. Plot the autocorrelation function and identify the period.

In [ ]:
def acf(x, max_lag):
    x = np.asarray(x, dtype=float)
    x_centered = x - x.mean()
    denom = np.sum(x_centered ** 2)
    vals = []
    for h in range(max_lag + 1):
        num = np.sum(x_centered[:len(x)-h] * x_centered[h:])
        vals.append(num / denom)
    return np.array(vals)

acf_vals = acf(x, max_lag=36)

plt.figure(figsize=(9, 4))
plt.stem(range(len(acf_vals)), acf_vals, basefmt=" ")
plt.xlabel("Lag")
plt.ylabel("ACF")
plt.title("Autocorrelation of a synthetic seasonal series")
plt.grid(True)
plt.show()

**Answer:** The ACF has visible peaks near lags 12, 24, and 36, which strongly suggests a seasonal period equal to 12.

#### Applications to datasets

You may use one or more of the following datasets:

- a synthetic seasonal series that you generate yourself,
- the `co2` dataset from `statsmodels`,
- the `sunspots` dataset from `statsmodels`,
- any other univariate time series dataset approved by the TA.

For the `co2` dataset, you may first resample or interpolate missing values if needed.

**Task 7:** Choose at least two datasets and do the following for each of them:

1. choose a reasonable period $p$,
2. compute the moving-average trend,
3. compute the seasonal component and the remainder,
4. plot the original series together with its components.

Comment on whether the additive decomposition seems appropriate.

In [ ]:
import statsmodels.api as sm

co2 = sm.datasets.co2.load_pandas().data["co2"].interpolate().to_numpy()
co2 = co2[~np.isnan(co2)]

sunspots = sm.datasets.sunspots.load_pandas().data["SUNACTIVITY"].to_numpy()

def plot_decomposition(x, p, q, title):
    trend = moving_average_zero_padding(x, q=q)
    pattern, seasonal, remainder = seasonal_decomposition_additive(x, trend, p=p)

    fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)
    axes[0].plot(x)
    axes[0].set_title(f"{title}: original")
    axes[1].plot(trend)
    axes[1].set_title("Trend")
    axes[2].plot(seasonal)
    axes[2].set_title("Seasonal")
    axes[3].plot(remainder)
    axes[3].set_title("Remainder")
    for ax in axes:
        ax.grid(True)
    plt.tight_layout()
    plt.show()

plot_decomposition(co2[:300], p=12, q=13, title="CO2 (first 300 observations)")
plot_decomposition(sunspots, p=11, q=11, title="Sunspots")

**Answer:** The additive decomposition is quite reasonable for the CO2 series, where a smooth trend and a recurring seasonal component are visible. For sunspots, the decomposition is less convincing: the oscillations are less regular and the amplitude is not especially stable, so a simple additive seasonal model is more questionable.

**Task 8:** For the same datasets, compute and plot the autocorrelation function.

Use the autocorrelation plots to guess the period of the seasonal component. Compare your conclusion with the period used in the decomposition.

In [ ]:
acf_co2 = acf(co2[:300], max_lag=60)
acf_sunspots = acf(sunspots, max_lag=40)

plt.figure(figsize=(9, 4))
plt.stem(range(len(acf_co2)), acf_co2, basefmt=" ")
plt.title("ACF of CO2")
plt.xlabel("Lag")
plt.ylabel("ACF")
plt.grid(True)
plt.show()

plt.figure(figsize=(9, 4))
plt.stem(range(len(acf_sunspots)), acf_sunspots, basefmt=" ")
plt.title("ACF of sunspots")
plt.xlabel("Lag")
plt.ylabel("ACF")
plt.grid(True)
plt.show()

**Answer:** For CO2, the ACF shows clear repeated peaks compatible with a period around 12. This agrees with the period used in the decomposition. For sunspots, the ACF suggests a longer oscillatory structure, but it is less sharply periodic than CO2, so the estimated period is only approximate.

**Task 9:** If you have some time left, try one or more of the following extensions:

- compare different moving-average window sizes,
- compare additive decomposition on a clearly non-seasonal dataset and on a strongly seasonal dataset,
- investigate what happens when the period is misspecified,
- compare your autocorrelation implementation with a library implementation.

**Possible extension comments:**

- A larger moving-average window gives a smoother trend but may oversmooth rapid changes.
- If the period is misspecified, the estimated seasonal component becomes distorted and the remainder keeps visible structure.
- Comparing with a library implementation is a useful way to validate the code written from scratch.